# 01 — Forward Pipeline: Codebase → Program Graph

This notebook walks through COGANT's **forward** pipeline: point it at a Python repository, build a typed `ProgramGraph`, and inspect the resulting nodes, edges, and role distribution.

It uses the `calculator` fixture that ships with COGANT. The goal is to let you poke at the graph interactively before moving on to GNN translation (notebook 02) and reverse synthesis (notebooks 03–04).

**Run from the `cogant/` directory** (one level above `docs/`), e.g. with:
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/01_forward_pipeline.ipynb
```

## 1. Imports and fixture resolution

We prefer `tests/fixtures/calculator/` if it exists and otherwise fall back to the canonical `examples/control_positive/calculator/` directory that ships with every COGANT checkout. Both contain a single Python module with classes, methods, and state mutations — enough to exercise every stage of the pipeline.

In [1]:
from __future__ import annotations

from pathlib import Path

import cogant
from cogant.api.session import Session

print(f"COGANT version: {cogant.__version__}")
print(f"Rust backend available: {cogant._RUST_AVAILABLE}")

COGANT version: 0.5.0
Rust backend available: True


In [2]:
# Resolve the calculator fixture. Prefer tests/fixtures/calculator; fall back
# to the always-present examples directory.
CANDIDATE_FIXTURES = [
    Path("tests/fixtures/calculator/"),
    Path("examples/control_positive/calculator/"),
]

FIXTURE: Path | None = None
for candidate in CANDIDATE_FIXTURES:
    if candidate.exists() and candidate.is_dir():
        FIXTURE = candidate.resolve()
        break

if FIXTURE is None:
    raise FileNotFoundError(
        f"No calculator fixture found. Looked in: {[str(p) for p in CANDIDATE_FIXTURES]}.\n"
        "Run this notebook from the cogant/ directory."
    )

print(f"Using fixture: {FIXTURE}")
print(f"Python files: {sorted(p.name for p in FIXTURE.rglob('*.py'))}")

Using fixture: /Users/4d/Documents/GitHub/template/projects_in_progress/cogant/cogant/examples/control_positive/calculator
Python files: ['calculator.py']


## 2. Run the forward pipeline

The `Session` object is COGANT's high-level entry point. `build_graph()` automatically runs the prerequisite stages: ingest → static analysis → normalize → graph. The returned dict is a JSON-friendly serialisation of the underlying `ProgramGraph`.

In [3]:
session = Session(repo_path=FIXTURE)
graph = session.build_graph()

nodes = graph.get("nodes", {})
edges = graph.get("edges", {})

print(f"Graph type: {graph.get('type')}")
print(f"Node count: {len(nodes)}")
print(f"Edge count: {len(edges)}")
print(f"Languages : {graph.get('metadata', {}).get('languages')}")

Graph type: program_graph
Node count: 12
Edge count: 25
Languages : ['python']


## 3. Role distribution

Each node has a `kind` (module, class, method, function, …). Counting kinds gives you a quick structural summary of the repository.

In [4]:
from collections import Counter

role_counts = Counter(n.get("kind", "unknown") for n in nodes.values())
print("Role distribution:")
for role, count in role_counts.most_common():
    print(f"  {role:<12} {count}")

Role distribution:
  method       10
  module       1
  class        1


In [5]:
# Pie chart of node-kind distribution.
import matplotlib
matplotlib.use("Agg")  # headless-safe; swap for %matplotlib inline in Jupyter
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
if role_counts:
    labels, sizes = zip(*role_counts.most_common())
    ax.pie(sizes, labels=labels, autopct="%1.0f%%", startangle=90)
    ax.set_title(f"Node roles in {FIXTURE.name}")
else:
    ax.text(0.5, 0.5, "(empty graph)", ha="center", va="center")
    ax.set_axis_off()

plt.tight_layout()
fig

<Figure size 500x500 with 1 Axes>

## 4. First five nodes as a DataFrame

Pandas gives a nicer view of the node table. We pick a handful of common columns; the rest are still accessible via `nodes[node_id]`.

In [6]:
try:
    import pandas as pd
except ImportError:
    print("pandas is not installed. Run: uv pip install pandas")
    pd = None  # type: ignore[assignment]

if pd is not None:
    rows = []
    for nid, node in list(nodes.items())[:5]:
        rows.append({
            "id": nid,
            "kind": node.get("kind"),
            "name": node.get("name"),
            "qualified_name": node.get("qualified_name"),
            "path": node.get("path"),
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

              id   kind            name                        qualified_name          path
c17ee576e914a0dd module      calculator                            calculator calculator.py
dfeb6c98e617f4d4  class      Calculator                 calculator.Calculator calculator.py
4327d190c2a7211b method        __init__        calculator.Calculator.__init__ calculator.py
b73f6e0374171462 method     input_digit     calculator.Calculator.input_digit calculator.py
45a37d3b19755461 method input_operation calculator.Calculator.input_operation calculator.py


## 5. Smoke assertion

If the graph has zero nodes, something is broken upstream (ingest, parsing, or the fixture itself). Fail loudly so CI catches it.

In [7]:
node_count = len(nodes)
assert node_count > 0, f"Expected a non-empty ProgramGraph, got {node_count} nodes"
print(f"OK — scan produced {node_count} nodes and {len(edges)} edges.")

OK — scan produced 12 nodes and 25 edges.
